Our package provides data access in a Python programming environment.

Here, we will start a Clustering analysis for the Pancreatic ductal adenocarcinoma (pdac).

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

# from gpnotebook.tools.standard_imports import *
import os, re,sys
import yaml
import pandas as pd
import numpy as np


In [2]:
# project_dir = r"/Users/yingweihu/Documents/GitHub/glycoproteinnotebook-private/data/v1/projects/PDAC_P_PDC000271"
project_dir = r"/Users/yingweihu/Documents/GitHub/glycoproteinnotebook-private/data/v1/projects/OV_G_JHU_PDC000250"
data_dir = os.path.join(project_dir,"matrix")
meta_dir = os.path.join(project_dir,"meta")
job_dir = os.path.join(project_dir,"precomputed","cluster")
if not os.path.exists(job_dir):
    os.mkdir(job_dir)

In [3]:
data_path = os.path.join(data_dir, "DIG_nglycoform-peptide_matrix-abundances-MD_norm.tsv")
data_df = pd.read_csv(data_path,sep="\t", index_col = [0,1,2,3])
data_df

Intensity.Reference  \
Site                                               Gene     Sequence                        Glycan                            
ENSP00000296504@78                                 SAP30    AAGNASFSK                       N4H5F0S1G0            15.000674   
ENSP00000361900@13                                 TOMM34   AAGNESFRNGQYAEASALYGR           N2H5F1S0G0            13.140487   
                                                                                            N3H5F0S0G0            14.269510   
ENSP00000262776@541                                LGALS3BP AAIPSALDTNSSK                   N2H3F0S0G0            14.332238   
                                                                                            N2H5F0S0G0            16.535804   
...                                                                                                                     ...   
ENSP00000376802@267;ENSP00000386094@267            SERPINA1 YLGNATAIFFLPDEGKLQHLENELTHDIITK N7H8F5S2G0            13.026964   
ENSP00000400031@77;ENSP00000368793@77              MTRF1    YMENLSK                         N4H5F1S0G0            14.486279   
ENSP00000497141@157;ENSP00000375678@173            KLK14    YPASLQCVNINISPDEVCQK            N2H4F0S0G0            16.177539   
ENSP00000413130@46;ENSP00000438497@134;ENSP0000... GNS      YPHNHHVVNNTLEGNCSSK             N6H9F1S2G0            13.476075   
                                                                                            N8H8F2S2G0            11.923064   

                                                                                                          pool_01  \
Site                                               Gene     Sequence                        Glycan                  
ENSP00000296504@78                                 SAP30    AAGNASFSK                       N4H5F0S1G0  15.000674   
ENSP00000361900@13                                 TOMM34   AAGNESFRNGQYAEASALYGR           N2H5F1S0G0  13.140487   
                                                                                            N3H5F0S0G0  14.269510   
ENSP00000262776@541                                LGALS3BP AAIPSALDTNSSK                   N2H3F0S0G0  14.332238   
                                                                                            N2H5F0S0G0  16.535804   
...                                                                                                           ...   
ENSP00000376802@267;ENSP00000386094@267            SERPINA1 YLGNATAIFFLPDEGKLQHLENELTHDIITK N7H8F5S2G0        NaN   
ENSP00000400031@77;ENSP00000368793@77              MTRF1    YMENLSK                         N4H5F1S0G0        NaN   
ENSP00000497141@157;ENSP00000375678@173            KLK14    YPASLQCVNINISPDEVCQK            N2H4F0S0G0        NaN   
ENSP00000413130@46;ENSP00000438497@134;ENSP0000... GNS      YPHNHHVVNNTLEGNCSSK             N6H9F1S2G0        NaN   
                                                                                            N8H8F2S2G0        NaN   

                                                                                                        15OV001_T_01  \
Site                                               Gene     Sequence                        Glycan                     
ENSP00000296504@78                                 SAP30    AAGNASFSK                       N4H5F0S1G0     15.636230   
ENSP00000361900@13                                 TOMM34   AAGNESFRNGQYAEASALYGR           N2H5F1S0G0     13.737074   
                                                                                            N3H5F0S0G0     15.124858   
ENSP00000262776@541                                LGALS3BP AAIPSALDTNSSK                   N2H3F0S0G0     14.326325   
                                                                                            N2H5F0S0G0     15.824359   
...                                                                                       

In [4]:
meta_path= os.path.join(meta_dir, "OV_meta.txt")
meta_df = pd.read_csv(meta_path,sep="\t",header=[0,1])
meta_df

,case_id,Age,Sex,Tumor_Size_cm,Histologic_Grade,Tumor_necrosis,Path_Stage_pT,Path_Stage_pN,Stage,BMI,Tobacco_smoking_history,mutation_table
,data_type,CON,BIN,CON,ORD,BIN,ORD,ORD,ORD,CON,ORD,BIN
0,01OV007,68,Female,NaN,G3 Poorly differentiated,NaN,NaN,NaN,Stage IV,NaN,NaN,1.0
1,01OV017,56,Female,NaN,G3 Poorly differentiated,NaN,NaN,NaN,Stage III,NaN,NaN,1.0
2,01OV018,44,Female,NaN,G3 Poorly differentiated,NaN,NaN,NaN,Stage III,NaN,NaN,1.0
3,01OV023,58,Female,NaN,G3 Poorly differentiated,NaN,NaN,NaN,Stage III,NaN,NaN,1.0
4,01OV026,77,Female,NaN,G3 Poorly differentiated,NaN,NaN,NaN,Stage III,NaN,NaN,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
78,26OV002,57,Female,NaN,G3 Poorly differentiated,NaN,NaN,NaN,Stage III,NaN,NaN,0.0
79,26OV008,65,Female,NaN,G3 Poorly differentiated,NaN,NaN,NaN,Stage IV,NaN,NaN,1.0
80,26OV009,60,Female,NaN,G3 Poorly differentiated,NaN,NaN,NaN,Stage III,NaN,NaN,1.0


In [5]:
meta_cols = ['case_id','Sex','Stage']
meta2 = meta_df.loc[:,meta_cols]
meta2.columns = ['Sample.ID'] + meta_cols[1:]
meta2

,Sample.ID,Sex,Stage
0,01OV007,Female,Stage IV
1,01OV017,Female,Stage III
2,01OV018,Female,Stage III
3,01OV023,Female,Stage III
4,01OV026,Female,Stage III
...,...,...,...
78,26OV002,Female,Stage III
79,26OV008,Female,Stage IV
80,26OV009,Female,Stage III
81,26OV011,Female,Stage III


In [6]:
meta2.head(17)

,Sample.ID,Sex,Stage
0,01OV007,Female,Stage IV
1,01OV017,Female,Stage III
2,01OV018,Female,Stage III
3,01OV023,Female,Stage III
4,01OV026,Female,Stage III
5,01OV029,Female,Stage III
6,01OV030,Female,Stage III
7,01OV039,Female,Stage III
8,01OV041,Female,Stage III
9,01OV047,Female,Stage III


In [7]:
head_cols = ['Site', 'Gene', 'Sequence', 'Glycan', 'Intensity.Reference']
samples = [i for i in data_df.columns.values if i not in head_cols]
samples = [i for i in samples if i.split('_')[0] in list(meta2['Sample.ID']) and i.split('_')[1] == 'T']
len(samples)

68

In [8]:
rows = []
for sample in samples:
    key = sample.split('_')[0]
    row = meta2[meta2['Sample.ID']==key].iloc[0]
    row['Sample.ID'] = sample
    rows.append(row)
meta3 = pd.DataFrame(rows)

In [9]:
meta3.head(17)

,Sample.ID,Sex,Stage
57,15OV001_T_01,Female,Stage III
63,17OV014_T_01,Female,Stage III
44,04OV048_T_01,Female,Stage III
25,04OV008_T_01,Female,Stage III
74,17OV039_T_01,Female,Stage III
42,04OV044_T_01,Female,Stage III
7,01OV039_T_02,Female,Stage III
8,01OV041_T_02,Female,Stage III
76,18OV001_T_02,Female,Stage I
41,04OV040_T_02,Female,Stage III


In [10]:
meta3 = meta3.replace(np.nan,'NA')

In [11]:
top_ann_data_path = os.path.join(job_dir,'top_ann_data.tsv')
meta3.to_csv(top_ann_data_path, sep="\t", index=False)

Top annotation settings.

In [12]:

top_ann_settings = {
    'Sex': {
        'Male': 'blue',
        'Female': 'red',
        'NA': 'grey',
    },
    'Stage': {
        'Stage I': 'blue',
        'Stage II': 'green',
        'Stage III': 'orange',
        'Stage IV': 'red',
        'NA': 'grey'
    },

}
top_ann_settings_path = os.path.join(job_dir,'top_ann_settings.yml')
with open(top_ann_settings_path,'w') as f:
    yaml.dump(top_ann_settings,f,default_flow_style=False)

In [13]:
data_df.head(2)

,,,,Intensity.Reference,pool_01,15OV001_T_01,11OV009_T_01,17OV014_T_01,04OV048_T_01,02OV001_T_01,15OV001_N_01,04OV008_T_01,17OV039_T_01,...,pool_13,26OV008_T_13,OTS5776_F_13,01OV029_T_13,17OV030_T_13,17OV017_T_13,20OV005_T_13,04OV053_T_13,01OV029_N_13,17OV003_N_13
Site,Gene,Sequence,Glycan,,,,,,,,,,,,,,,,,,,,,
ENSP00000296504@78,SAP30,AAGNASFSK,N4H5F0S1G0,15.000674,15.000674,15.636230,14.971136,14.210948,14.064614,14.662376,16.473918,14.859474,14.132572,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ENSP00000361900@13,TOMM34,AAGNESFRNGQYAEASALYGR,N2H5F1S0G0,13.140487,13.140487,13.737074,12.183324,11.414014,11.926131,12.622150,13.554111,12.594419,11.884627,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
data_df.shape

(7185, 131)

In [15]:
samples = meta3['Sample.ID'].to_list()

In [16]:
len(samples)

68

In [17]:
df2 = data_df.loc[:,samples].dropna()

In [18]:
df2.shape

(427, 68)

In [19]:
from scipy.stats import variation
rows = []
for index,row in df2.iterrows():
    rows.append([variation([np.power(2,i) for i in list(row)])])
cv_df = pd.DataFrame(rows,columns=['cv'],index= df2.index)

glycopeptides = cv_df[cv_df['cv']>0.25].index

data2 = df2[df2.index.isin(glycopeptides)]
glycopeptides =  [f'{site}@{gene}@{seq}@{glycan}' for site,gene,seq,glycan in glycopeptides]
data2.index = glycopeptides
tumor_expression_path = os.path.join(job_dir,'expression_data.tsv')
data2.to_csv(tumor_expression_path,sep='\t',index=True)

In [20]:
data2.shape

(410, 68)

Extract tumor samples from glycopeptide expression data based on pathological status,

calculates the coefficient of variation (CV) for each glycopeptide, selects glycopeptides with CV greater than 0.25.

Map glcopeptides with cv>0.25 in tumor patients with glycan type.

In [21]:
import re,os, sys

def decide_glycan_type(g):
    m = re.finditer("([A-Z])([\d]+)", g)
    y = [(i.group(1), int(i.group(2))) for i in m]
    d = dict(y)
    glycan_type = "Other"
    if d["N"] == 2 and d["H"] >= 5 and d["F"] == 0 and d["S"] == 0 and d["G"] == 0:
        glycan_type = "HM"
    elif d["N"] >= 2 and d["H"] >= 3 and d["F"] > 0 and d["S"] == 0:
        glycan_type = "only_F"
    elif d["N"] >= 2 and d["H"] >= 3 and d["S"] > 0 and d["F"] == 0:
        glycan_type = "only_S"
    elif d["N"] >= 2 and d["H"] >= 3 and d["S"] > 0 and d["F"] > 0:
        glycan_type = "F+S"
    return glycan_type


In [22]:
# left annotation
# from gpnotebook.tools.glycan import decide_glycan_type

glycan_type_map = dict(zip(glycopeptides,[decide_glycan_type(i) for i in glycopeptides]))
  
left_ann_data_path =  os.path.join(job_dir,'left_annotation_data.tsv')
rows = []
for i in glycan_type_map:
    rows.append([i,glycan_type_map[i]])
left_ann_data = pd.DataFrame(rows,columns=['Glycopeptide','GlycanType'])
left_ann_data.to_csv(left_ann_data_path,sep="\t",index=False)

In [23]:
left_ann_data

,Glycopeptide,GlycanType
0,ENSP00000262776@541@LGALS3BP@AAIPSALDTNSSK@N2H...,HM
1,ENSP00000262776@541@LGALS3BP@AAIPSALDTNSSK@N2H...,HM
2,ENSP00000262776@541@LGALS3BP@AAIPSALDTNSSK@N2H...,HM
3,ENSP00000262776@541@LGALS3BP@AAIPSALDTNSSK@N3H...,only_F
4,ENSP00000262776@541@LGALS3BP@AAIPSALDTNSSK@N3H...,F+S
...,...,...
405,ENSP00000376802@267;ENSP00000386094@267@SERPIN...,F+S
406,ENSP00000376802@267;ENSP00000386094@267@SERPIN...,only_F
407,ENSP00000376802@267;ENSP00000386094@267@SERPIN...,only_F
408,ENSP00000308541@135;ENSP00000433907@135@F2@YPH...,only_S


Map glycan types with colors.

In [24]:

# left annotation settings, including color, order
left_ann_settings_path = os.path.join(job_dir,'left_annotation_settings.yml')
left_ann_settings = {
    "glycan_type_index" :{
    "HM": 1,
    "only_F":2,
    "only_S":3,
    "F+S":4,
    "Other":5
    },
    "glycan_type_color" : {
        "HM": 'green',
    "only_F": 'red',
    "only_S": 'purple',
    "F+S": 'orange',
    "Other": 'grey'
}
}
with open(left_ann_settings_path,'w') as f:
    yaml.dump(left_ann_settings,f,default_flow_style=False)
    

Parameters for NMF clustering.

In [25]:
nmf_parameters_path = os.path.join(job_dir, 'nmf_parameters.yml')
nmf_parameters = {
    'k_range': {
        'min': 3,
        'max': 5,
    },
    'test':{
        'nruns': 50
    },
    'opt_k':{
        'nruns': 500,
        'predefined': 0,
        'value': 4,
        'feature_prob': 0.8
    }
}
with open(nmf_parameters_path,'w') as f:
    yaml.dump(nmf_parameters,f,default_flow_style=False)

Generate a YAML configuration file (nmf_configs.yml) containing paths to various data required for NMF clustering.

In [26]:
config_data = {
    'input': {
        'expression_data': tumor_expression_path,
        'left_annotation_data': left_ann_data_path ,
        'left_annotation_settings': left_ann_settings_path,
        'top_annotation_data': top_ann_data_path,
        'top_annotatin_settings': top_ann_settings_path,
        'nmf_parameters': nmf_parameters_path
    },
    'output':{
        'out_dir': job_dir
    }
}
nmf_configs_path = os.path.join(job_dir,'nmf_configs.yml')
with open(nmf_configs_path,'w') as f:
    yaml.dump(config_data,f,default_flow_style=False)